In [11]:
#[CELL 1] STUDENT CONFIGURATION
import csv, json, shutil
from collections import Counter
from pathlib import Path

LAST_NAME = "Dantes"
STUDENT_ID = "TUPM-26-7955"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

RECORD_LIMIT = 1000 + ID_SUM          # records to process
TARGET_CLASS = (SEED_DIGIT % 4) + 2   # 2 -> 2xx, 3 -> 3xx, 4 -> 4xx, 5 -> 5xx

print("Seed:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Records to process:", RECORD_LIMIT)
print("Target status class:", f"{TARGET_CLASS}xx")

Seed: 5
ID Sum: 34
Records to process: 1034
Target status class: 3xx


In [12]:
#[CELL 2] WORKSPACE AND FILE PATHS
BASE = Path(".")
DATA_DIR = BASE / "data"
OUTPUT_DIR = BASE / "output"
BACKUP_DIR = BASE / "backup"

for folder in (DATA_DIR, OUTPUT_DIR, BACKUP_DIR):
    folder.mkdir(exist_ok=True)

dataset_path = DATA_DIR / "weblog.csv"
filtered_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_ex1_filtered.csv"
json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_ex1_summary.json"
backup_path = BACKUP_DIR / json_path.name

print("Workspace:", BASE.resolve())
print("Dataset exists:", dataset_path.exists())

Workspace: C:\Users\anica\Lab6_File_Handling_Dantes\Lab6_File_Handling_Additional_Exercises
Dataset exists: True


In [13]:
#[CELL 3] READ AND PROCESS THE LOG LINE BY LINE
records_processed = 0
invalid_records = 0
status_counts = Counter()    # every processed record
target_counts = Counter()    # status codes inside the target class
url_counts = Counter()       # URLs inside the target class
selected = []                # (IP, Time, URL, Status)

with open(dataset_path, "r", encoding="utf-8", newline="") as file:
    reader = csv.reader(file)
    header = [h.strip().lower() for h in next(reader, [])]

    i_ip = header.index("ip")
    i_time = header.index("time")
    i_url = header.index("url")
    i_status = header.index("staus") if "staus" in header else header.index("status")
    needed = max(i_ip, i_time, i_url, i_status) + 1

    for row in reader:
        if records_processed >= RECORD_LIMIT:
            break
        records_processed += 1

        if len(row) < needed:                 # missing fields
            invalid_records += 1
            continue

        ip = row[i_ip].strip()
        timestamp = row[i_time].strip()
        url = row[i_url].strip()

        try:
            status = int(row[i_status].strip())
        except ValueError:                    # non-numeric status
            invalid_records += 1
            continue

        status_counts[status] += 1

        if status // 100 == TARGET_CLASS:
            selected.append((ip, timestamp, url, status))
            target_counts[status] += 1
            url_counts[url] += 1

top5 = url_counts.most_common(5)

print("Records processed:", records_processed)
print("Invalid records skipped:", invalid_records)
print("Records in target class:", len(selected))

Records processed: 1034
Invalid records skipped: 0
Records in target class: 166


In [14]:
#[CELL 4] EXPORT FILTERED CSV
with open(filtered_path, "w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["IP", "Time", "URL", "Status"])
    writer.writerows(selected)

print("Filtered CSV created:", filtered_path)
print("Rows written:", len(selected))

Filtered CSV created: output\Dantes_TUPM-26-7955_ex1_filtered.csv
Rows written: 166


In [15]:
#[CELL 5] WRITE JSON SUMMARY
summary = {
    "last_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "seed_digit": SEED_DIGIT,
    "id_sum": ID_SUM,
    "records_processed": records_processed,
    "invalid_records": invalid_records,
    "target_status_class": f"{TARGET_CLASS}xx",
    "records_in_target_class": len(selected),
    "count_per_status_code_all": dict(sorted(status_counts.items())),
    "count_per_status_code_target": dict(sorted(target_counts.items())),
    "five_most_requested_urls": [{"url": u, "count": c} for u, c in top5],
}

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("JSON summary created:", json_path)

JSON summary created: output\Dantes_TUPM-26-7955_ex1_summary.json


In [16]:
#[CELL 6] BACKUP AND VERIFY
shutil.copy2(json_path, backup_path)

backup_ok = (backup_path.exists()
             and backup_path.stat().st_size == json_path.stat().st_size)

print("Original:", json_path)
print("Backup:", backup_path)
print("Backup exists:", backup_path.exists())
print("Backup verification:", "VERIFIED" if backup_ok else "FAILED")

Original: output\Dantes_TUPM-26-7955_ex1_summary.json
Backup: backup\Dantes_TUPM-26-7955_ex1_summary.json
Backup exists: True
Backup verification: VERIFIED


In [17]:
#[CELL 7] FINAL OUTPUT
print("=== EXERCISE 1: WEB SERVER ACCESS LOG ANALYSIS ===")
print("Records processed       :", records_processed)
print("Target status class     :", f"{TARGET_CLASS}xx")
print("Records in target class :", len(selected))
print("Count per status code   :", dict(sorted(status_counts.items())))
print("Five most requested URLs:")
for rank, (u, c) in enumerate(top5, 1):
    print(f"  {rank}. {u}  ({c})")
print("Filtered CSV file       :", filtered_path)
print("JSON summary file       :", json_path)
print("Backup verification     :", "VERIFIED" if backup_ok else "FAILED")

=== EXERCISE 1: WEB SERVER ACCESS LOG ANALYSIS ===
Records processed       : 1034
Target status class     : 3xx
Records in target class : 166
Count per status code   : {200: 855, 206: 8, 302: 145, 304: 21, 404: 5}
Five most requested URLs:
  1. GET / HTTP/1.1  (47)
  2. POST /process.php HTTP/1.1  (35)
  3. GET /contestproblem.php?name=RUET%20OJ%20Server%20Testing%20Contest HTTP/1.1  (34)
  4. GET /home.php HTTP/1.1  (9)
  5. GET /logout.php HTTP/1.1  (7)
Filtered CSV file       : output\Dantes_TUPM-26-7955_ex1_filtered.csv
JSON summary file       : output\Dantes_TUPM-26-7955_ex1_summary.json
Backup verification     : VERIFIED
